# PCG-MAS — Frontier Cloud Runner (16-cell scaffold, 2-cell execution)

Full 8-dataset x 2-frontier-model grid (16 cells) wired for PCG-MAS R1-R5 + additional
paper artifacts + all 6 SOTA baselines. A budget gate executes only the 2 manuscript-critical
cells; the other 14 are scaffolded and dry-runnable but never spent on.

- **Datasets (8):** fever, hotpotqa, twowiki, toolbench, pubmedqa, tatqa, weblinx, synthetic
- **Frontier models (2):** Llama-3.3-70B (quantized local), deepseek-v3 (API-only)
- **Execution targets (2):** `toolbench:Llama-3.3-70B`, `weblinx:deepseek-v3`

Secrets via Colab Secrets only (HF_TOKEN, ANTHROPIC_API_KEY). Never paste tokens inline.


## 1 - Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/pcg-submission'
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)


## 2 - Clone / update repo


In [ ]:
%cd /content
![ -d pcg ] || git clone https://github.com/anonymous-submission/proof-carrying-multi-agents.git pcg
%cd /content/pcg
!git pull --ff-only || true
!git log --oneline -1


## 3 - Secrets (Colab Secrets only)
Set `HF_TOKEN` and `ANTHROPIC_API_KEY` in the Colab Secrets panel (key icon, left).
HF_TOKEN must have access to gated meta-llama/Llama-3.3-70B-Instruct.


In [ ]:
import os
from google.colab import userdata
for k in ('HF_TOKEN','ANTHROPIC_API_KEY','DEEPSEEK_API_KEY'):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception as e:
        print(f'{k}: not set in Secrets ({e})')
print('HF_TOKEN        :', 'set' if os.environ.get('HF_TOKEN') else 'MISSING')
print('ANTHROPIC_KEY   :', 'set' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING (only for ShieldAgent)')
print('DEEPSEEK_API_KEY:', 'set' if os.environ.get('DEEPSEEK_API_KEY') else 'MISSING (needed for deepseek-v3 cell)')


## 4 - HF cache on Drive (download weights once)


In [ ]:
import os
HF_CACHE = f'{DRIVE_ROOT}/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_DATASETS_CACHE'] = f'{HF_CACHE}/datasets'
os.environ['TRANSFORMERS_CACHE'] = f'{HF_CACHE}/transformers'
print('HF_HOME =', os.environ['HF_HOME'])


## 5 - Install pinned deps (matches local multi-agents venv)
Same pinset as local: transformers 4.46.x, torch 2.5.x, tokenizers <0.21, hf_hub <1.0.
Plus GPU-only inference extras (bitsandbytes for 4-bit, vllm if the GPU supports it).


In [ ]:
import subprocess, sys
BASE = ['transformers>=4.45,<4.50','tokenizers>=0.20,<0.21','huggingface_hub>=0.23,<1.0',
        'accelerate>=1.0,<2.0','safetensors>=0.4,<1.0','datasets>=2.20,<5',
        'sentence-transformers>=3.0,<6','rank-bm25>=0.2','faiss-cpu>=1.7',
        'numpy>=1.26,<3','scipy>=1.11,<2','scikit-learn>=1.3,<2','pandas>=2.0,<4',
        'tiktoken>=0.7','openai>=1.30','anthropic>=0.40','langgraph>=0.2,<2',
        'langchain-core>=0.3,<2','matplotlib','seaborn','plotly','loguru','tqdm','pyyaml',
        'bitsandbytes>=0.43']
!pip -q install {' '.join(BASE)} 2>&1 | tail -3
!pip -q install -e . 2>&1 | tail -3
# torch/torchvision pinned AFTER editable install (pyproject can pull newer torch)
!pip -q install 'torch>=2.4,<2.6' 'torchvision==0.20.1' 2>&1 | tail -2


## 6 - GPU check + adaptive 70B backend selection
Picks vLLM if a big GPU is present, else 4-bit NF4 local, else API fallback.


In [ ]:
import torch, subprocess
has_cuda = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_cuda else 'none'
gpu_mem_gb = (torch.cuda.get_device_properties(0).total_memory/1e9) if has_cuda else 0
print(f'CUDA={has_cuda}  GPU={gpu_name}  mem={gpu_mem_gb:.0f}GB')

# Decide 70B strategy by available VRAM
if gpu_mem_gb >= 75:
    LLAMA70B_MODE = 'vllm'        # A100 80GB / H100: full or AWQ via vLLM
elif gpu_mem_gb >= 38:
    LLAMA70B_MODE = 'nf4_4bit'    # 40GB A100: 4-bit NF4 local
else:
    LLAMA70B_MODE = 'api'         # fallback: HF Inference API
print('Llama-3.3-70B backend mode:', LLAMA70B_MODE)
print('deepseek-v3 backend mode  : api (hf_inference, always)')


## 7 - The 16-cell grid + budget gate
Full scaffold defined; only ALLOWLIST cells actually execute.


In [ ]:
DATASETS = ['fever','hotpotqa','twowiki','toolbench','pubmedqa','tatqa','weblinx','synthetic']
FRONTIER_MODELS = ['Llama-3.3-70B','deepseek-v3']

FULL_GRID = [f'{d}:{m}' for m in FRONTIER_MODELS for d in DATASETS]  # 16 cells
assert len(FULL_GRID) == 16

# Budget gate: only these execute. Everything else is scaffolded, never spent on.
ALLOWLIST = ['toolbench:Llama-3.3-70B', 'weblinx:deepseek-v3']

print(f'Full grid: {len(FULL_GRID)} cells')
for c in FULL_GRID:
    flag = 'EXECUTE' if c in ALLOWLIST else 'scaffold'
    print(f'  [{flag:8s}] {c}')


## 8 - Output tree on Drive (matches local dataset:model layout)
Results land under Drive in the same structure local expects, for clean sync-back.


In [ ]:
OUT_ROOT = f'{DRIVE_ROOT}/results'
import os
os.makedirs(f'{OUT_ROOT}/tables/csv/experiment_json', exist_ok=True)
os.makedirs(f'{OUT_ROOT}/baselines', exist_ok=True)
# Symlink repo results -> Drive so runners write straight to Drive
import subprocess
subprocess.run(['rm','-rf','/content/pcg/results'])
os.symlink(OUT_ROOT, '/content/pcg/results')
print('repo results/ -> ', os.readlink('/content/pcg/results'))


## 9 - Backend env for the runners
The PCG runners read backend.kind from config; we override per-cell via env the
runner respects. For 70B local we use hf_local; for deepseek-v3 we use hf_inference.


In [ ]:
import os
# Per-cell backend resolution.
#   deepseek-v3      -> official DeepSeek API (platform.deepseek.com)
#   Llama-3.3-70B    -> local 4-bit on A100 (hf_local) if GPU>=38GB, else API fallback
#   everything else  -> hf_local
def pcg_backend_for(cell):
    model = cell.split(':',1)[1]
    if model == 'deepseek-v3':
        return 'deepseek'
    if model == 'Llama-3.3-70B':
        return 'hf_local' if LLAMA70B_MODE != 'api' else 'hf_inference'
    return 'hf_local'
for c in ALLOWLIST:
    print(f'{c:32s} -> {pcg_backend_for(c)}')


## 10 - PREFLIGHT: dry n=3 on one allowlist cell
Smoke the full additional-artifacts pipeline at tiny n before spending budget.


In [ ]:
CELL = 'toolbench:Llama-3.3-70B'
BACKEND = pcg_backend_for(CELL)
!cd /content/pcg && PYTHONPATH=src \
  bash scripts/runs/run_additional_paper_artifacts.sh \
  --cells {CELL} --n-examples 3 --seed 0 --backend {BACKEND} 2>&1 | tail -40


## 10b - PREFLIGHT: DeepSeek API canary (n=3)
Cheap canary for the `weblinx:deepseek-v3` cell. Verifies DEEPSEEK_API_KEY auth
and that the official DeepSeek API serves requests BEFORE the n=30 run. If this
errors (401 / quota / model id), STOP and fix the key — do not run cell 12+.


In [ ]:
import os
if not os.environ.get('DEEPSEEK_API_KEY'):
    print('SKIP: DEEPSEEK_API_KEY not set in Secrets — set it before the weblinx:deepseek-v3 run')
else:
    CELL = 'weblinx:deepseek-v3'
    BACKEND = pcg_backend_for(CELL)   # -> 'deepseek'
    print(f'DeepSeek canary: {CELL} (backend={BACKEND}, n=3)')
    !cd /content/pcg && PYTHONPATH=src \
      bash scripts/runs/run_additional_paper_artifacts.sh \
      --cells {CELL} --n-examples 3 --seed 0 --backend {BACKEND} 2>&1 | tail -40


## 11 - EXECUTE: full PCG additional-artifacts on the 2 allowlist cells
Manuscript-grade n. Adjust N as budget allows.


In [ ]:
N = 30
for CELL in ALLOWLIST:
    BACKEND = pcg_backend_for(CELL)
    print(f'\n===== PCG additional artifacts: {CELL} (backend={BACKEND}, n={N}) =====')
    !cd /content/pcg && PYTHONPATH=src \
      bash scripts/runs/run_additional_paper_artifacts.sh \
      --cells {CELL} --n-examples {N} --seed 0 --backend {BACKEND} 2>&1 | tail -30


## 12 - EXECUTE: core R1-R5 on the 2 allowlist cells
The standard matrix experiments (separate from additional artifacts).


In [ ]:
N = 30
for CELL in ALLOWLIST:
    ds, model = CELL.split(':',1)
    BACKEND = pcg_backend_for(CELL)
    for exp in ['r1_checkability','r2_redundancy','r3_responsibility','r4_risk_privacy','r5_overhead']:
        extra = '--k-values 1 2 4' if exp == 'r2_redundancy' else ''
        print(f'\n=== {exp}: {CELL} ===')
        !cd /content/pcg && PYTHONPATH=src python scripts/experiments/run_{exp}.py \
          --dataset {ds} --model {model} --backend {BACKEND} --n-examples {N} --seeds 0 {extra} 2>&1 | tail -8


## 13 - EXECUTE: SOTA baselines on the 2 allowlist cells
5 twins via --pairs/--backend-mode; ShieldAgent separately via --cell + Anthropic.


In [ ]:
PAIRS = ','.join(ALLOWLIST)
N = 30
# backend-mode for SOTA twins: hf_local for 70B (if not api), else openai is N/A here;
# frontier cells use hf_local when weights load, otherwise the twin's api path.
for method in ['agentrr','verimap','atlasprism','pcnrec','clbc']:
    print(f'\n===== SOTA {method}: {PAIRS} =====')
    !cd /content/pcg && PYTHONPATH=src python scripts/baselines/{method}/run_{method}_r1_r5.py \
      --pairs {PAIRS} --seeds 0 --n-examples {N} --backend-mode hf_local 2>&1 | tail -12


### ShieldAgent (Anthropic-backed, separate CLI)
Requires ANTHROPIC_API_KEY. Runs per-cell with explicit jsonl paths.


In [ ]:
import os
if not os.environ.get('ANTHROPIC_API_KEY'):
    print('SKIP ShieldAgent: ANTHROPIC_API_KEY not set')
else:
    for CELL in ALLOWLIST:
        slug = CELL.replace(':','__')
        base = f'{OUT_ROOT}/baselines/shieldagent/{slug}'
        !mkdir -p {base}
        !cd /content/pcg && PYTHONPATH=src python scripts/baselines/shieldagent/run_shieldagent_r1_r5_comparative.py \
          --cell {CELL} \
          --input-jsonl {base}/input.jsonl \
          --output-jsonl {base}/output.jsonl \
          --metrics-json {base}/metrics.json \
          --r2-json {base}/r2.json --r3-json {base}/r3.json --r4-json {base}/r4.json \
          2>&1 | tail -12


## 14 - Zip results for local sync-back
Download the zip, unzip into local repo `results/`, then rebuild figures/tables locally.


In [ ]:
import time
STAMP = time.strftime('%Y%m%d-%H%M%S')
ZIP = f'{DRIVE_ROOT}/pcg_frontier_results_{STAMP}.zip'
!cd {DRIVE_ROOT} && zip -r -q {ZIP} results
print('wrote', ZIP)
print('Sync-back: unzip into local repo root so it merges results/tables/csv/experiment_json/')
from google.colab import files
# files.download(ZIP)   # uncomment to pull straight to your machine
